# Durable Vintage CORE reference evaluation

Runs full **Original**, **Filtered**, and **Restyled** Vintage CORE for:

- `modern-d24`
- `gpt1900-d34` (Michael Hla)

Select an **A100 GPU** and add `HF_TOKEN` to Colab Secrets before running all cells. The token must read the Vintage CORE dataset and write to `jbduran/think.nano`.

Every completed bundle JSON is written to Google Drive and uploaded immediately to Hugging Face. If Colab disconnects, reopen this notebook and run all cells again; completed bundles are restored and skipped. Only the bundle active during a hard runtime termination may need to restart.

In [ ]:
import os, platform, shutil, subprocess, sys
from pathlib import Path

import torch
from google.colab import drive, userdata

MODELS = ["modern-d24", "gpt1900-d34"]
BUNDLES = ["original", "filtered", "restyled"]
EVALUATOR_COMMIT = "82b7e92adf04aac6418b29e6bbca7ddfd479c462"
RESULT_REPO = "jbduran/think.nano"
RESULT_PREFIX = "evaluations/vintage-core-v1.0.0"

assert torch.cuda.is_available(), "Select a GPU runtime first."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB)")
if "A100" not in gpu.name:
    print("WARNING: This notebook is sized and timed for an A100.")
print(f"Python {platform.python_version()} | torch {torch.__version__} | CUDA {torch.version.cuda}")

token = userdata.get("HF_TOKEN")
assert token, "Add HF_TOKEN to Colab Secrets and enable notebook access."
os.environ["HF_TOKEN"] = token

drive.mount("/content/drive")
RESULTS_ROOT = Path("/content/drive/MyDrive/vintage-core-results-v1.0.0")
CACHE_ROOT = Path("/content/vintage-core-cache")
REPO_DIR = Path("/content/think.nano-vintage-core")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent results:", RESULTS_ROOT)

In [ ]:
REPO_URL = "https://github.com/zachnorton14/think.nano.git"
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", EVALUATOR_COMMIT, "--depth", "1"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", EVALUATOR_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == EVALUATOR_COMMIT

requirements = REPO_DIR / "dev/vintage_core_colab/requirements.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
print("Pinned evaluator ready:", actual_commit)

In [ ]:
import hashlib
import json
import queue
import threading
import time

from huggingface_hub import HfApi, hf_hub_download

EVALUATOR_DIR = REPO_DIR / "dev/vintage_core_colab"
EVALUATOR = EVALUATOR_DIR / "vintage_core_eval.py"
sys.path.insert(0, str(EVALUATOR_DIR))
import vintage_core_eval as vc

registry = vc.read_json(EVALUATOR_DIR / "models.json")["models"]
expected = {
    "modern-d24": ("ChrisMcCormick/nanochat-d24-2026-02-02", "2ccf42323ff3bedcc986191d688e5827f33c237c", "7ac837cff8efc0e85502e2b3a934a35e2d937b8d"),
    "gpt1900-d34": ("mhla/gpt1900-d34-22btok", "d6330f9f0a17ce13da36fb951d7987bb03e6fbd0", None),
}
for model_id, (artifact_repo, artifact_revision, runtime_revision) in expected.items():
    entry = registry[model_id]
    assert entry["artifact_repo"] == artifact_repo
    assert entry["artifact_revision"] == artifact_revision
    assert entry["runtime"].get("revision") == runtime_revision

api = HfApi(token=token)
remote_files = set(api.list_repo_files(RESULT_REPO, repo_type="model"))
uploaded_hashes = set()

def valid_bundle(path, model_id, bundle):
    if not path.is_file():
        return False
    try:
        record = json.loads(path.read_text())
    except (OSError, ValueError, json.JSONDecodeError):
        return False
    return (record.get("model") == model_id and record.get("bundle") == bundle
            and record.get("max_per_task") == -1 and record.get("core_metric") is not None)

def restore_remote(model_id, output_dir):
    prefix = f"{RESULT_PREFIX}/{model_id}/"
    restored = 0
    for repo_path in sorted(path for path in remote_files if path.startswith(prefix)):
        cached = hf_hub_download(RESULT_REPO, repo_path, repo_type="model", token=token)
        destination = output_dir / repo_path.removeprefix(prefix)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cached, destination)
        restored += 1
    print(f"{model_id}: restored {restored} files from Hugging Face")

def upload_completed_json(model_id, output_dir):
    for bundle in BUNDLES:
        path = output_dir / f"{bundle}.json"
        if not valid_bundle(path, model_id, bundle):
            continue
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        marker = (model_id, bundle, digest)
        if marker in uploaded_hashes:
            continue
        try:
            api.upload_file(
                repo_id=RESULT_REPO, repo_type="model", path_or_fileobj=str(path),
                path_in_repo=f"{RESULT_PREFIX}/{model_id}/{bundle}.json",
                commit_message=f"Save {model_id} {bundle} Vintage CORE result",
            )
        except Exception as exc:
            print(f"UPLOAD RETRY NEEDED for {model_id}/{bundle}: {type(exc).__name__}: {exc}", flush=True)
            continue
        uploaded_hashes.add(marker)
        print(f"PERSISTED IMMEDIATELY: {model_id}/{bundle}.json", flush=True)

def run_and_persist(model_id):
    output_dir = RESULTS_ROOT / model_id
    output_dir.mkdir(parents=True, exist_ok=True)
    restore_remote(model_id, output_dir)
    upload_completed_json(model_id, output_dir)
    command = [
        sys.executable, "-u", str(EVALUATOR), "--model", model_id,
        "--bundles", ",".join(BUNDLES), "--output-dir", str(output_dir),
        "--cache-dir", str(CACHE_ROOT), "--max-per-task", "-1",
    ]
    print("Running:", " ".join(command), flush=True)
    process = subprocess.Popen(
        command, cwd=str(REPO_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    output_queue = queue.Queue()
    def pump_output():
        for line in process.stdout:
            output_queue.put(line)
        output_queue.put(None)
    threading.Thread(target=pump_output, daemon=True).start()
    started = time.monotonic()
    while True:
        try:
            line = output_queue.get(timeout=30)
        except queue.Empty:
            upload_completed_json(model_id, output_dir)
            completed = [bundle for bundle in BUNDLES if valid_bundle(output_dir / f"{bundle}.json", model_id, bundle)]
            try:
                gpu_status = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader,nounits"],
                    text=True, timeout=5,
                ).strip().replace("\n", "; " )
            except Exception as exc:
                gpu_status = f"unavailable ({type(exc).__name__})"
            elapsed = (time.monotonic() - started) / 60
            print(
                f"[heartbeat {model_id}: {elapsed:.1f} min | GPU util%, memory MiB: {gpu_status} | "
                f"completed: {completed or 'none'}]", flush=True,
            )
            continue
        if line is None:
            break
        print(line, end="", flush=True)
        upload_completed_json(model_id, output_dir)
    returncode = process.wait()
    upload_completed_json(model_id, output_dir)
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    api.upload_folder(
        repo_id=RESULT_REPO, repo_type="model", folder_path=str(output_dir),
        path_in_repo=f"{RESULT_PREFIX}/{model_id}",
        commit_message=f"Finalize {model_id} Vintage CORE tables",
    )
    print(f"COMPLETE AND PERSISTED: {model_id}", flush=True)

print("Durability helpers ready.")

In [ ]:
for model_id in MODELS:
    print(f"\n{'=' * 80}\n{model_id}\n{'=' * 80}", flush=True)
    run_and_persist(model_id)

print("\nAll reference evaluations are complete.")
print("Google Drive:", RESULTS_ROOT)
print(f"Hugging Face: https://huggingface.co/{RESULT_REPO}/tree/main/{RESULT_PREFIX}")

In [ ]:
import pandas as pd
from IPython.display import display

frames = []
for model_id in MODELS:
    summary = RESULTS_ROOT / model_id / "summary.csv"
    if summary.is_file():
        frame = pd.read_csv(summary)
        frame.insert(0, "model", model_id)
        frames.append(frame)
if frames:
    display(pd.concat(frames, ignore_index=True).style.format({
        "native_core": "{:.6f}", "common_20_core": "{:.6f}",
        "runtime_seconds": "{:.1f}",
    }))
else:
    print("No completed summaries yet. Rerun the evaluation cell to continue.")